In [38]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [40]:
import patchtst_dataset

print("patchtst_dataset loaded from:")
print(patchtst_dataset.__file__)

print("\nAvailable names:")
print([x for x in dir(patchtst_dataset) if not x.startswith("_")])

patchtst_dataset loaded from:
/content/patchtst_dataset.py

Available names:
['Any', 'DATE_METADATA_COLS', 'DEFAULT_PATCHTST_DIR', 'DataLoader', 'Dataset', 'PatchTSTDataset', 'Path', 'STATIC_METADATA_COLS', 'annotations', 'create_patchtst_dataloaders', 'load_npz_mmap', 'main', 'np', 'torch', 'verify_patchtst_dataloaders', 'zipfile']


In [42]:
import inspect
from patchtst_dataset import create_patchtst_dataloaders

print(inspect.signature(create_patchtst_dataloaders))

(patchtst_dir: 'Path | str' = PosixPath('C:\\CODE\\favorita_inventory_pipeline\\outputs\\patchtst'), batch_size: 'int' = 64, patch_size: 'int' = 7, stride: 'int' = 7, channel_names: 'list[str] | None' = None, num_workers: 'int' = 0, pin_memory: 'bool' = False) -> 'tuple[dict[str, DataLoader], dict[str, PatchTSTDataset]]'


In [43]:
from patchtst_dataset import create_patchtst_dataloaders

patchtst_dir = "/content/patchtst"

loaders, datasets = create_patchtst_dataloaders(
    patchtst_dir=patchtst_dir,
    batch_size=64,
    patch_size=7,
    stride=7,
    channel_names=["unit_sales_scaled", "onpromotion"],
    num_workers=2,
    pin_memory=True
)

train_loader = loaders["train"]
val_loader = loaders["validation"]
test_loader = loaders["test"]

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

Train samples: 624451
Validation samples: 3299
Test samples: 3237


In [44]:
# Get one batch from the training loader
batch = next(iter(train_loader))

print("Batch keys:", batch.keys())

for key, value in batch.items():
    if hasattr(value, "shape"):
        print(f"{key}: {value.shape}")

Batch keys: dict_keys(['past_x', 'patches', 'y', 'sequence_idx', 'y_raw', 'store_nbr', 'item_nbr', 'family', 'class', 'perishable', 'city', 'state', 'type', 'cluster', 'input_start_date', 'input_end_date', 'target_start_date', 'target_end_date'])
past_x: torch.Size([64, 28, 2])
patches: torch.Size([64, 2, 4, 7])
y: torch.Size([64, 7])
sequence_idx: torch.Size([64])
y_raw: torch.Size([64, 7])
store_nbr: torch.Size([64])
item_nbr: torch.Size([64])
class: torch.Size([64])
perishable: torch.Size([64])
cluster: torch.Size([64])


In [45]:
import torch
import torch.nn as nn


class PatchTST(nn.Module):

    def __init__(
        self,
        input_channels=2,
        num_patches=4,
        patch_size=7,
        d_model=64,
        n_heads=4,
        num_layers=3,
        dim_feedforward=128,
        dropout=0.1,
        forecast_horizon=7
    ):
        super().__init__()

        self.input_channels = input_channels
        self.num_patches = num_patches
        self.patch_size = patch_size
        self.d_model = d_model

        # Convert each 7-day patch into a 64-dimensional vector
        self.patch_embedding = nn.Linear(patch_size, d_model)

        # Learn the position of each patch
        self.positional_embedding = nn.Parameter(
            torch.zeros(1, num_patches, d_model)
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Use ALL channels × ALL patches
        head_input_size = input_channels * num_patches * d_model

        self.forecast_head = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(head_input_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, forecast_horizon)
        )

    def forward(self, patches):

        # patches: [B, C, N_patches, patch_size]
        B, C, N, P = patches.shape

        # Process every channel independently through the Transformer
        x = patches.reshape(B * C, N, P)

        x = self.patch_embedding(x)

        x = x + self.positional_embedding

        x = self.transformer_encoder(x)

        # Restore channels
        x = x.reshape(B, C, N, self.d_model)

        # Forecast using all channel/patch representations
        output = self.forecast_head(x)

        return output


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PatchTST().to(device)

print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Device: cuda
Parameters: 167751


/tmp/ipykernel_658/3394204212.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(


In [46]:
model.eval()

with torch.no_grad():
    batch = next(iter(train_loader))

    patches = batch["patches"].to(device)

    predictions = model(patches)

print("Input:", patches.shape)
print("Output:", predictions.shape)

Input: torch.Size([64, 2, 4, 7])
Output: torch.Size([64, 7])


In [47]:
import torch.optim as optim

criterion = nn.MSELoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

num_epochs = 30
patience = 5

best_val_loss = float("inf")
epochs_without_improvement = 0

checkpoint_path = "/content/patchtst_best_model.pth"

print("Optimizer: AdamW")
print("Learning rate: 0.001")
print("Weight decay: 0.0001")
print("Maximum epochs:", num_epochs)
print("Early stopping patience:", patience)

Optimizer: AdamW
Learning rate: 0.001
Weight decay: 0.0001
Maximum epochs: 30
Early stopping patience: 5


In [48]:
import time

for epoch in range(1, num_epochs + 1):

    start_time = time.time()

    # --------------------
    # Training
    # --------------------
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        patches = batch["patches"].to(device)
        targets = batch["y"].to(device)

        optimizer.zero_grad()

        predictions = model(patches)
        loss = criterion(predictions, targets)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * patches.size(0)

    train_loss /= len(train_loader.dataset)

    # --------------------
    # Validation
    # --------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            patches = batch["patches"].to(device)
            targets = batch["y"].to(device)

            predictions = model(patches)
            loss = criterion(predictions, targets)

            val_loss += loss.item() * patches.size(0)

    val_loss /= len(val_loader.dataset)

    epoch_time = time.time() - start_time

    # --------------------
    # Save best model
    # --------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(model.state_dict(), checkpoint_path)

        status = "✓ best"
    else:
        epochs_without_improvement += 1
        status = f"no improvement ({epochs_without_improvement}/{patience})"

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {epoch_time:.1f}s | "
        f"{status}"
    )

    # --------------------
    # Early stopping
    # --------------------
    if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered.")
        break

print("\nTraining complete.")
print("Best validation loss:", best_val_loss)
print("Best model saved to:", checkpoint_path)

Epoch 01 | Train Loss: 0.222593 | Val Loss: 0.169678 | Time: 193.0s | ✓ best
Epoch 02 | Train Loss: 0.214936 | Val Loss: 0.166403 | Time: 191.0s | ✓ best
Epoch 03 | Train Loss: 0.213305 | Val Loss: 0.165268 | Time: 191.1s | ✓ best
Epoch 04 | Train Loss: 0.212235 | Val Loss: 0.163416 | Time: 191.0s | ✓ best
Epoch 05 | Train Loss: 0.211631 | Val Loss: 0.165719 | Time: 193.1s | no improvement (1/5)
Epoch 06 | Train Loss: 0.210970 | Val Loss: 0.168056 | Time: 190.4s | no improvement (2/5)
Epoch 07 | Train Loss: 0.210776 | Val Loss: 0.169762 | Time: 191.6s | no improvement (3/5)
Epoch 08 | Train Loss: 0.210519 | Val Loss: 0.165135 | Time: 193.1s | no improvement (4/5)
Epoch 09 | Train Loss: 0.209990 | Val Loss: 0.164666 | Time: 191.1s | no improvement (5/5)

Early stopping triggered.

Training complete.
Best validation loss: 0.1634158307077235
Best model saved to: /content/patchtst_best_model.pth


In [49]:
import os
import torch

checkpoint_path = "/content/patchtst_best_model.pth"

print("File exists:", os.path.exists(checkpoint_path))
print("File size:", round(os.path.getsize(checkpoint_path) / (1024**2), 2), "MB")

checkpoint = torch.load(checkpoint_path, map_location="cpu")

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", checkpoint.keys())

File exists: True
File size: 0.66 MB
Checkpoint type: <class 'collections.OrderedDict'>
Checkpoint keys: odict_keys(['positional_embedding', 'patch_embedding.weight', 'patch_embedding.bias', 'transformer_encoder.layers.0.self_attn.in_proj_weight', 'transformer_encoder.layers.0.self_attn.in_proj_bias', 'transformer_encoder.layers.0.self_attn.out_proj.weight', 'transformer_encoder.layers.0.self_attn.out_proj.bias', 'transformer_encoder.layers.0.linear1.weight', 'transformer_encoder.layers.0.linear1.bias', 'transformer_encoder.layers.0.linear2.weight', 'transformer_encoder.layers.0.linear2.bias', 'transformer_encoder.layers.0.norm1.weight', 'transformer_encoder.layers.0.norm1.bias', 'transformer_encoder.layers.0.norm2.weight', 'transformer_encoder.layers.0.norm2.bias', 'transformer_encoder.layers.1.self_attn.in_proj_weight', 'transformer_encoder.layers.1.self_attn.in_proj_bias', 'transformer_encoder.layers.1.self_attn.out_proj.weight', 'transformer_encoder.layers.1.self_attn.out_proj.bias

In [50]:
from google.colab import files

files.download("/content/patchtst_best_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [51]:
# Load the best saved model and evaluate on the test set

import torch
import numpy as np

# Recreate the corrected PatchTST architecture
model = PatchTST(
    input_channels=2,
    num_patches=4,
    patch_size=7,
    d_model=64,
    n_heads=4,
    num_layers=3,
    dim_feedforward=128,
    dropout=0.1,
    forecast_horizon=7
).to(device)

# Load best checkpoint
checkpoint_path = "/content/patchtst_best_model.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

model.eval()

predictions = []
targets = []
raw_targets = []

with torch.no_grad():
    for batch in test_loader:
        patches = batch["patches"].to(device)

        output = model(patches)

        predictions.append(output.cpu().numpy())
        targets.append(batch["y"].numpy())
        raw_targets.append(batch["y_raw"].numpy())

predictions = np.concatenate(predictions, axis=0)
targets = np.concatenate(targets, axis=0)
raw_targets = np.concatenate(raw_targets, axis=0)

print("Predictions shape:", predictions.shape)
print("Targets shape:", targets.shape)
print("Raw targets shape:", raw_targets.shape)

/tmp/ipykernel_658/3394204212.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(


Predictions shape: (3237, 7)
Targets shape: (3237, 7)
Raw targets shape: (3237, 7)


In [52]:
# Convert predictions back to original sales units and calculate metrics

import joblib
import numpy as np

# Load the saved scaler
scaler_data = joblib.load("/content/target_sales_scaler.joblib")
scaler = scaler_data["target_scaler"]

# Inverse transform:
# scaled value -> log1p(sales) -> actual sales
pred_log = scaler.inverse_transform(predictions.reshape(-1, 1)).reshape(predictions.shape)
pred_sales = np.expm1(pred_log)

# Actual test sales
actual_sales = raw_targets

# Prevent tiny numerical negatives
pred_sales = np.maximum(pred_sales, 0)

# Flatten for overall metrics
y_true = actual_sales.flatten()
y_pred = pred_sales.flatten()

# MAE
mae = np.mean(np.abs(y_true - y_pred))

# RMSE
rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

# WAPE
wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

# sMAPE
denominator = np.abs(y_true) + np.abs(y_pred)
mask = denominator != 0

smape = (
    np.mean(
        2 * np.abs(y_true[mask] - y_pred[mask])
        / denominator[mask]
    ) * 100
)

print("PatchTST Test Metrics")
print("---------------------")
print(f"MAE   : {mae:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"WAPE  : {wape:.2f}%")
print(f"sMAPE : {smape:.2f}%")

PatchTST Test Metrics
---------------------
MAE   : 13.2549
RMSE  : 31.7976
WAPE  : 29.95%
sMAPE : 40.11%
